<a href="https://colab.research.google.com/github/madhavvcs24-arch/AI-LAB/blob/main/FOL_Unification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import re

# Check whether a term is a variable
def is_variable(x):
    return isinstance(x, str) and x[0].isupper()


# Apply substitutions to a term
def substitute(term, subst):
    if isinstance(term, str):
        if term in subst:
            return substitute(subst[term], subst)
        return term

    name, args = term
    return (name, tuple(substitute(arg, subst) for arg in args))


# Check occurs condition
def occurs(var, term, subst):
    term = substitute(term, subst)

    if var == term:
        return True

    if isinstance(term, tuple):
        return any(occurs(var, arg, subst) for arg in term[1])

    return False


# Unification function
def unify(x, y, subst):

    x = substitute(x, subst)
    y = substitute(y, subst)

    # If both terms are identical
    if x == y:
        return subst

    # If x is a variable
    if is_variable(x):
        if occurs(x, y, subst):
            return None

        subst[x] = y
        return subst

    # If y is a variable
    if is_variable(y):
        if occurs(y, x, subst):
            return None

        subst[y] = x
        return subst

    # If both are compound expressions
    if isinstance(x, tuple) and isinstance(y, tuple):

        # Check predicate/function names and argument count
        if x[0] != y[0] or len(x[1]) != len(y[1]):
            return None

        # Unify corresponding arguments
        for a, b in zip(x[1], y[1]):
            subst = unify(a, b, subst)

            if subst is None:
                return None

        return subst

    # Otherwise unification fails
    return None


# Convert input expression into terms
def parse(expression):

    tokens = re.findall(r'[A-Za-z_][A-Za-z0-9_]*|[(),]', expression)
    pos = [0]

    def parse_term():

        name = tokens[pos[0]]
        pos[0] += 1

        if pos[0] < len(tokens) and tokens[pos[0]] == '(':

            pos[0] += 1
            args = []

            while True:
                args.append(parse_term())

                if tokens[pos[0]] == ',':
                    pos[0] += 1
                else:
                    break

            pos[0] += 1

            return (name, tuple(args))

        return name

    return parse_term()


# Display terms properly
def display(term):

    if isinstance(term, str):
        return term

    name, args = term

    return name + "(" + ",".join(display(a) for a in args) + ")"


# Solve one unification problem
def solve(expression1, expression2):

    term1 = parse(expression1)
    term2 = parse(expression2)

    subst = unify(term1, term2, {})

    if subst is None:
        print("MGU: FAILURE")
        return

    if not subst:
        print("MGU: NIL")
        return

    print("MGU: {", end="")

    results = []

    for var in subst:
        value = substitute(subst[var], subst)
        results.append(var + " = " + display(value))

    print(", ".join(results), "}")


# Main program
print("UNIFICATION ALGORITHM")
print("=" * 45)

# Question 1
print("\nQuestion 1:")
solve("P(f(a),g(y))", "P(X,X)")

# Question 2
print("\nQuestion 2:")
solve("Q(a,g(x,a),f(y))",
      "Q(a,g(f(b),a),x)")

# Question 3
print("\nQuestion 3:")
solve("P(b,x,f(g(z)))",
      "P(z,f(y),f(y))")


UNIFICATION ALGORITHM

Question 1:
MGU: FAILURE

Question 2:
MGU: FAILURE

Question 3:
MGU: FAILURE
